In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 3.2 Symmetric Matrices and the Spectral Theorem

In [ ]:
from ecp.style import header, use_style

use_style()
header(
    volume="Volume III — Eigenvalues and Spectral Theory",
    number="3.2",
    title="Symmetric Matrices and the Spectral Theorem",
    blurb="Impose one condition — A equals its own transpose — and every "
    "difficulty of the previous notebook disappears at once: real eigenvalues, "
    "orthonormal eigenvectors, perfect conditioning, and eigenvalues you can "
    "localise without computing anything.",
    difficulty="advanced",
    estimate="105–135 min",
)

## Notebook overview

[§3.1](eigenvalues-diagonalization.ipynb) left a list of things that can go
wrong: eigenvalues that turn out complex, eigenvectors that are not orthogonal,
an eigenvector matrix so badly conditioned that {eq}`eq-eig-diag` stops meaning
anything, and a matrix with too few eigenvectors to diagonalize at all. Every
one of those disappears under a single hypothesis, $A = A^{\top}$.

The **spectral theorem** says a real symmetric matrix has real eigenvalues and
an *orthonormal* basis of eigenvectors, always. So $X$ becomes an orthogonal
$Q$ with $\operatorname{cond}(Q) = 1$ exactly, $X^{-1}$ becomes $Q^{\top}$, and
$A = Q\Lambda Q^{\top}$ is a decomposition into a rotation, a scaling along the
axes, and the reverse rotation. Nothing is defective, nothing is
ill-conditioned, and nothing is complex.

What follows is a run of results that are unusually strong for how cheap they
are. The **Rayleigh quotient** turns the extreme eigenvalues into an
optimisation problem, and the eigenvectors into its stationary points.
**Courant–Fischer** extends that to every eigenvalue in between. **Cauchy
interlacing** says deleting the last row and column of a symmetric matrix
produces eigenvalues that slot between the originals, every time. **Weyl's
inequality** says a perturbation of norm $\|E\|$ moves no eigenvalue further
than $\|E\|$ — so the symmetric eigenvalue problem is *perfectly conditioned*,
which is not true of any other problem in this course. And **Gershgorin's
theorem** localises every eigenvalue by reading the entries off the matrix,
with no computation at all.

The notebook ends by settling a loose end from [§3.1](eigenvalues-diagonalization.ipynb),
where the Rayleigh quotient converged at the same linear rate as the vector.
Under symmetry it converges *quadratically*, and the constant turns out to be
exactly the gap between the top two eigenvalues.

> **How to read a check.** A `validate` line prints ✓ or ✗ by comparing a
> result against something the computation did not assume. A ✗ flags a
> mismatch to investigate, never a verdict on its own.

> **Scope.** Strang {cite}`strang2023` Chapter 6; Horn and Johnson
> {cite}`horn2013` Chapters 4 and 6 for interlacing, Weyl and Gershgorin in
> full; Trefethen and Bau {cite}`trefethen1997` Lecture 24; Parlett's
> *The Symmetric Eigenvalue Problem* is the monograph.

## Theory in brief

### The theorem

If $A \in \mathbb{R}^{n\times n}$ satisfies $A = A^{\top}$, then all its
eigenvalues are real and it has an orthonormal basis of eigenvectors. Writing
those in the columns of $Q$,

```{math}
:label: eq-spec-decomposition
A = Q\Lambda Q^{\top},
\qquad Q^{\top}Q = QQ^{\top} = I,
\qquad \Lambda = \operatorname{diag}(\lambda_1 \le \dots \le \lambda_n).
```

Both halves are short to see. Reality: if $A\mathbf{x} = \lambda\mathbf{x}$
with $\mathbf{x}$ possibly complex, then
$\bar{\mathbf{x}}^{\top}A\mathbf{x} = \lambda\,\bar{\mathbf{x}}^{\top}\mathbf{x}$,
and the left side equals its own conjugate because $A$ is real symmetric, while
$\bar{\mathbf{x}}^{\top}\mathbf{x} > 0$; so $\lambda = \bar{\lambda}$.
Orthogonality: if $A\mathbf{x} = \lambda\mathbf{x}$ and $A\mathbf{y} =
\mu\mathbf{y}$ with $\lambda \ne \mu$, then $\lambda\,\mathbf{y}^{\top}\mathbf{x}
= \mathbf{y}^{\top}\!A\mathbf{x} = \mu\,\mathbf{y}^{\top}\mathbf{x}$, forcing
$\mathbf{y}^{\top}\mathbf{x} = 0$. Repeated eigenvalues need a little more care
and the result survives it.

Expanding {eq}`eq-spec-decomposition` column by column gives the form worth
carrying around,

```{math}
:label: eq-spec-projectors
A = \sum_{i=1}^{n} \lambda_i\,\mathbf{q}_i\mathbf{q}_i^{\top} ,
\qquad \sum_{i=1}^{n} \mathbf{q}_i\mathbf{q}_i^{\top} = I ,
```

a weighted sum of rank-1 orthogonal projectors: $A$ *is* a list of directions,
each with a number attached.

### The Rayleigh quotient

For $\mathbf{x} \ne \mathbf{0}$,

```{math}
:label: eq-spec-rayleigh
\rho(\mathbf{x}) = \frac{\mathbf{x}^{\top}\!A\mathbf{x}}{\mathbf{x}^{\top}\mathbf{x}} .
```

Writing $\mathbf{x} = \sum c_i\mathbf{q}_i$ and using orthonormality gives
$\rho(\mathbf{x}) = \sum \lambda_ic_i^2 / \sum c_i^2$: a **weighted average of
the eigenvalues**. Three consequences follow immediately. It is trapped in
$[\lambda_1, \lambda_n]$; it attains both ends, at $\mathbf{q}_1$ and
$\mathbf{q}_n$; and its gradient

```{math}
:label: eq-spec-gradient
\nabla\rho(\mathbf{x}) =
  \frac{2\bigl(A\mathbf{x} - \rho(\mathbf{x})\,\mathbf{x}\bigr)}
       {\mathbf{x}^{\top}\mathbf{x}}
```

vanishes exactly when $A\mathbf{x} = \rho(\mathbf{x})\mathbf{x}$ — that is,
exactly at the eigenvectors. Eigenvectors are the **stationary points** of
{eq}`eq-spec-rayleigh`, and that is a definition one can optimise against.

### Courant–Fischer

The middle eigenvalues get the same treatment through subspaces. With
$\lambda_1 \le \dots \le \lambda_n$,

```{math}
:label: eq-spec-minmax
\lambda_k = \min_{\substack{V \subseteq \mathbb{R}^n \\ \dim V = k}}
  \;\max_{\mathbf{0}\ne\mathbf{x}\in V} \rho(\mathbf{x}) ,
```

and the minimising subspace is $\operatorname{span}\{\mathbf{q}_1, \dots,
\mathbf{q}_k\}$. This characterises $\lambda_k$ without reference to any
eigenvector, which is what makes the next two results provable at all.

### Cauchy interlacing

Let $A_{n-1}$ be $A$ with its last row and column removed. Then

```{math}
:label: eq-spec-interlacing
\lambda_k(A_n) \;\le\; \lambda_k(A_{n-1}) \;\le\; \lambda_{k+1}(A_n) ,
\qquad k = 1, \dots, n-1 ,
```

so the $n-1$ eigenvalues of the submatrix slot into the gaps between the $n$
eigenvalues of the whole. It follows from {eq}`eq-spec-minmax` by noting that
subspaces of $\mathbb{R}^{n-1}$ are a subset of those of $\mathbb{R}^n$.

### Weyl, and perfect conditioning

For symmetric $A$ and $E$,

```{math}
:label: eq-spec-weyl
\bigl|\lambda_k(A + E) - \lambda_k(A)\bigr| \;\le\; \|E\|_2
\qquad\text{for every } k .
```

The condition number of the symmetric eigenvalue problem is therefore **1**:
an input perturbation of size $\epsilon$ moves the answer by at most $\epsilon$.
Nothing else in this course is that well behaved, and it is the reason
`eigh` is one of the most trustworthy routines in LAPACK. It is emphatically
*false* without symmetry, as [§3.5](schur-jordan-nonnormality.ipynb) will show.

### Gershgorin

Every eigenvalue of any square $A$ (symmetric or not) lies in the union of the
discs

```{math}
:label: eq-spec-gershgorin
D_i = \Bigl\{ z \in \mathbb{C} : |z - a_{ii}| \le R_i \Bigr\} ,
\qquad R_i = \sum_{j \ne i} |a_{ij}| .
```

The proof is two lines: if $A\mathbf{x} = \lambda\mathbf{x}$ and $i$ is the
index of the largest $|x_i|$, then row $i$ gives
$(\lambda - a_{ii})x_i = \sum_{j\ne i}a_{ij}x_j$, and dividing by $x_i$ bounds
the left side by $R_i$. It costs $O(n^2)$ reads and no arithmetic worth
counting, and when the discs are disjoint each contains exactly one eigenvalue.

---
## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from ecp import validate
from ecp import linalg as la
from ecp.style import use_style

use_style()
rng = np.random.default_rng(0)  # every random array below comes from this seed

EPS = np.finfo(float).eps
np.set_printoptions(precision=6, suppress=True, linewidth=120)

# The notebook's worked symmetric matrix, fixed here and restated in every
# exercise that uses it.
S = np.array([[4.0,  1.0, -2.0,  0.0,  1.0],
              [1.0,  5.0,  0.0, -1.0,  2.0],
              [-2.0, 0.0,  6.0,  1.0,  0.0],
              [0.0, -1.0,  1.0,  3.0, -1.0],
              [1.0,  2.0,  0.0, -1.0,  7.0]])

# A diagonally dominant matrix whose Gershgorin discs are disjoint.
S_DOM = np.diag([3.0, 10.0, 17.0, 24.0, 31.0]) + 0.4 * np.array(
    [[0.0, 1.0, -1.0, 0.0, 1.0],
     [1.0, 0.0, 1.0, -1.0, 0.0],
     [-1.0, 1.0, 0.0, 1.0, -1.0],
     [0.0, -1.0, 1.0, 0.0, 1.0],
     [1.0, 0.0, -1.0, 1.0, 0.0]])
S_DOM = (S_DOM + S_DOM.T) / 2.0


# Residuals of an eigendecomposition scale with the size of the matrix: a
# backward-stable routine returns eigenvalues accurate to about c * eps * ||S||,
# not to some absolute constant. Everything in this notebook that compares two
# computed spectra, or an eigenvalue against a Rayleigh quotient, is therefore
# gated against this rather than a bare number — including across two different
# LAPACK drivers, where the constant c is whatever the two implementations
# happen to differ by.
TOL = 100.0 * EPS * float(np.linalg.norm(S, 2))


def rayleigh(A, x):
    """The Rayleigh quotient of Eq. 4, a weighted average of the eigenvalues."""
    x = np.asarray(x, dtype=float)
    return float(x @ A @ x / (x @ x))


def rayleigh_grad(A, x):
    """The gradient of Eq. 4, given by Eq. 5; zero exactly at the eigenvectors."""
    x = np.asarray(x, dtype=float)
    return 2.0 * (A @ x - rayleigh(A, x) * x) / (x @ x)

## Exercise 1 — What symmetry buys, measured against what it replaces

{eq}`eq-spec-decomposition` makes four promises: real eigenvalues, orthonormal
eigenvectors, a reconstruction with $Q^{\top}$ in place of $X^{-1}$, and
$\operatorname{cond}(Q) = 1$. This exercise checks all four on

$$
S = \begin{bmatrix}
 4 & 1 & -2 & 0 & 1\\ 1 & 5 & 0 & -1 & 2\\ -2 & 0 & 6 & 1 & 0\\
 0 & -1 & 1 & 3 & -1\\ 1 & 2 & 0 & -1 & 7 \end{bmatrix},
$$

available as `S`, and compares against what
[§3.1](eigenvalues-diagonalization.ipynb) got from a non-symmetric matrix of
the same kind of size.

The right call is `np.linalg.eigh`, not `eig`. It **assumes** symmetry, reads
only the lower triangle by default, exploits that to run in roughly half the
time, returns real eigenvalues in **ascending order**, and returns an
orthonormal $Q$. Using `eig` on a symmetric matrix is not wrong, but it throws
away every one of those guarantees and returns complex dtypes for real answers.

**Part a)** Confirm `S` is symmetric with
`np.allclose(S, S.T, rtol=0, atol=0)` — exactly, not approximately, since it
was written down as an integer array.

**Part b)** Take `lam, Q = np.linalg.eigh(S)`. Report the eigenvalues and
confirm they come back in ascending order with `np.all(np.diff(lam) > 0)`.
They are $1.7420, 3.3150, 3.8300, 6.9401, 9.1730$.

**Part c)** Confirm orthonormality: $\|Q^{\top}Q - I\|_{\max} < 10^{-14}$, and
report $\operatorname{cond}(Q)$, which is 1 to within $10^{-13}$. Compare with
[§3.1](eigenvalues-diagonalization.ipynb)'s $\operatorname{cond}(X) = 2.51$ for
a non-symmetric matrix, and with its $9\times10^{15}$ for a defective one. An
orthogonal matrix cannot be badly conditioned.

**Part d)** Confirm the reconstruction {eq}`eq-spec-decomposition`:
$\|Q\Lambda Q^{\top} - S\|_{\max} < 10^{-13}$, using `Q @ np.diag(lam) @ Q.T`.
No inverse is formed anywhere, because the transpose *is* the inverse.

**Part e)** Confirm what `eig` gives on the same matrix. Report
`np.abs(np.linalg.eigvals(S).imag).max()`, which is exactly $0$, and confirm
the sorted real parts match `lam` to `TOL`$= 100\varepsilon\|S\|_2 =
2.0\times10^{-13}$ — a *stated* backward-error scale rather than a bare
number, because two different LAPACK drivers agree only to within their own
rounding, and how closely is not something to guess at. The general routine finds the
same answer; it simply cannot promise in advance that it will.

In [ ]:
# (solution hidden on the public site)


### Validation 1

$\operatorname{cond}(Q) = 1$ is the check worth pausing on: it is an *exact*
statement about an orthogonal matrix, and it is the single number that
separates {eq}`eq-spec-decomposition` from the general diagonalization of
[§3.1](eigenvalues-diagonalization.ipynb), where that number was free to be
anything at all.

In [ ]:
validate.check(
    np.allclose(S, S.T, rtol=0, atol=0) and bool(np.all(np.diff(lam) > 0)),
    "S is exactly symmetric and eigh returns eigenvalues in ascending order",
    f"eigenvalues {np.array2string(lam, precision=4)}, all distinct and sorted",
)
validate.close(
    Q.T @ Q, np.eye(5),
    "the eigenvectors are orthonormal: Q^T Q = I (Eq. 1)",
    rtol=0.0, atol=1e-13,
)
validate.close(
    np.array([cond_Q]), np.array([1.0]),
    "so cond(Q) = 1 exactly, which no general eigenvector matrix can promise",
    rtol=0.0, atol=1e-13,
)
validate.close(
    Q @ np.diag(lam) @ Q.T, S,
    "and A = Q Lambda Q^T reconstructs the matrix with no inverse (Eq. 1)",
    rtol=0.0, atol=TOL,
)
validate.check(
    imag_max == 0.0 and gen_gap < TOL,
    "np.linalg.eig finds the same real spectrum, it just cannot guarantee it",
    f"largest |Im lambda| = {imag_max:.1e}, agreement with eigh {gen_gap:.2e} "
    f"against the backward-error scale 100 eps ||S|| = {TOL:.2e}: eigh assumes "
    "symmetry, reads one triangle, and is about twice as fast",
)

## Exercise 2 — The matrix as a sum of projectors

{eq}`eq-spec-projectors` re-reads {eq}`eq-spec-decomposition` as a statement
about *structure*: a symmetric matrix is a set of orthogonal directions with a
number attached to each, and nothing else. Each $\mathbf{q}_i\mathbf{q}_i^{\top}$
is the rank-1 orthogonal projector onto its eigendirection, exactly the object
[§1.4](../01-matrices/four-subspaces.ipynb) built, and the projectors are
mutually orthogonal and sum to the identity — a **resolution of the identity**.

This is also where low-rank approximation begins. Keeping only the terms with
the largest $|\lambda_i|$ gives the best rank-$k$ symmetric approximation to
$A$, which Volume IV proves properly.

**Part a)** Build the five rank-1 projectors `P[i] = np.outer(Q[:, i], Q[:, i])`
and confirm each is idempotent, $P_i^2 = P_i$ to $10^{-14}$, and symmetric to
$10^{-15}$.

**Part b)** Confirm mutual orthogonality: $P_iP_j = 0$ to $10^{-14}$ for every
$i \ne j$, which is orthonormality of the $\mathbf{q}_i$ restated as a product
of matrices.

**Part c)** Confirm the resolution of the identity: $\sum_i P_i = I$ to
$10^{-14}$. Every vector splits into five orthogonal pieces, one per
eigendirection.

**Part d)** Confirm {eq}`eq-spec-projectors` itself:
$\sum_i \lambda_iP_i = S$ to $10^{-13}$.

**Part e)** Watch the sum accumulate. For $k = 1, \dots, 5$ form the partial
sum over the $k$ largest $|\lambda_i|$ and report
$\|S - \sum_{\text{top }k}\lambda_iP_i\|_{\max}$. It falls $5.215$, $2.611$,
$2.377$, $0.748$, $0$ — reaching exactly zero at $k = 5$ and not before,
because $S$ has full rank. Confirm the sequence is non-increasing and that the
final error is below $10^{-13}$.

**Part f)** Draw the five partial sums as heatmaps beside $S$ itself, with
`la.matrix_heatmap`, so the matrix can be seen assembling one direction at a
time.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 2

Idempotence, mutual orthogonality and the resolution of the identity are
checked separately, because together they say $\{P_i\}$ is a *complete
orthogonal family* — the statement {eq}`eq-spec-projectors` actually makes,
and a stronger one than the reconstruction alone.

In [ ]:
validate.check(
    idem < 1e-13 and symm < 1e-14,
    "each q_i q_i^T is a symmetric idempotent: an orthogonal projector (Eq. 2)",
    f"worst |P^2 - P| = {idem:.2e}, worst |P - P^T| = {symm:.2e}",
)
validate.check(
    cross < 1e-13,
    "and the five projectors are mutually orthogonal: P_i P_j = 0",
    f"largest entry of any P_i P_j with i != j is {cross:.2e}, which is "
    "orthonormality of the eigenvectors written as a matrix product",
)
validate.close(
    sum(P), np.eye(5),
    "they resolve the identity: sum_i q_i q_i^T = I (Eq. 2)",
    rtol=0.0, atol=1e-13,
)
validate.close(
    sum(lam[i] * P[i] for i in range(5)), S,
    "and sum_i lambda_i q_i q_i^T reproduces S (Eq. 2)",
    rtol=0.0, atol=TOL,
)
validate.check(
    all(a >= b - 1e-15 for a, b in zip(partial_err, partial_err[1:]))
    and partial_err[-1] < 1e-13,
    "the partial sums improve monotonically and are exact only at k = n",
    f"errors {[f'{e:.4f}' for e in partial_err]}: full rank means no rank-4 "
    "approximation can be exact",
)

## Exercise 3 — The Rayleigh quotient, and eigenvectors as stationary points

{eq}`eq-spec-rayleigh` converts an algebraic question into an optimisation one.
Because $\rho(\mathbf{x}) = \sum\lambda_ic_i^2/\sum c_i^2$ is a weighted average
of the eigenvalues with non-negative weights, it can never leave
$[\lambda_1, \lambda_n]$, and it reaches each end only by putting all the weight
on one eigenvector. The extreme eigenvalues are therefore the extreme values of
a smooth function on the unit sphere, and the extreme eigenvectors are where it
attains them.

The stronger statement is {eq}`eq-spec-gradient`: *every* eigenvector is a
stationary point, not only the extreme two. This exercise checks both the
bracket and the stationarity, and it does so in a way that keeps the exact
claims and the sampled evidence separate.

**Part a)** Draw $10^{5}$ random unit vectors in $\mathbb{R}^5$ as
`X = rng.standard_normal((100_000, 5))` normalised rowwise by
`np.linalg.norm(X, axis=1, keepdims=True)`, and evaluate
{eq}`eq-spec-rayleigh` on all of them at once with
`np.einsum("ij,jk,ik->i", X, S, X)`. Confirm the **exact** claim: every one of
the $10^{5}$ values lies in $[\lambda_1, \lambda_n] = [1.7420, 9.1730]$, to
$10^{-12}$. This is a theorem and admits no exceptions.

**Part b)** Report how close the sample gets to the ends: the largest value is
$9.1609$ against $\lambda_5 = 9.1730$ and the smallest is $1.7670$ against
$\lambda_1 = 1.7420$, so random sampling recovers the top eigenvalue to about
$0.1\%$ and the bottom to about $1.4\%$. Confirm both relative gaps are below
$5\%$, and note why they are not smaller: hitting a specific direction in five
dimensions by chance is hard, which is exactly why power iteration exists.

**Part c)** Confirm exact attainment at the eigenvectors: $\rho(\mathbf{q}_i)
= \lambda_i$ to $10^{-14}$ for all five. Sampling approaches the answer;
the eigenvector *is* the answer.

**Part d)** Confirm stationarity {eq}`eq-spec-gradient` with
`rayleigh_grad(S, Q[:, i])` for each $i$, checking
$\|\nabla\rho\| < 10^{-14}$ at every eigenvector, including the three
*interior* ones which are neither maxima nor minima but saddle points. Then
report $\|\nabla\rho\|$ at one random unit vector, which is about $6.2$ — not
small by any reading.

**Part e)** Draw the Rayleigh quotient of the $2\times2$ matrix
$S_2 = \left[\begin{smallmatrix}6&1\\1&5\end{smallmatrix}\right]$ around the
unit circle, as a function of the angle $\theta$, marking the four points
$\pm\mathbf{q}_1$, $\pm\mathbf{q}_2$. Confirm the curve's maximum and minimum
equal $\lambda_2 = 6.6180$ and $\lambda_1 = 4.3820$ to $10^{-6}$ over a
2001-point grid, and that the extrema occur at the eigenvector angles to within
one grid step.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 3

The bracket is gated as the exact theorem it is; the *approach* of the sample
to the ends is reported and gated only loosely, because how close $10^{5}$
random directions get is a fact about sampling in five dimensions and not about
the matrix. The stationarity check covers all five eigenvectors, including the
saddle points, which is the part a maximum-and-minimum reading would miss.

In [ ]:
validate.check(
    in_bracket,
    "every one of 100,000 random Rayleigh quotients lies in [lam_1, lam_n] (Eq. 4)",
    f"sampled range [{rq_samples.min():.6f}, {rq_samples.max():.6f}] inside "
    f"[{lam[0]:.6f}, {lam[-1]:.6f}]: a weighted average of the eigenvalues "
    "cannot escape them",
)
validate.check(
    abs(rq_samples.max() - lam[-1]) / abs(lam[-1]) < 5e-2
    and abs(rq_samples.min() - lam[0]) / abs(lam[0]) < 5e-2,
    "and the sample approaches both ends without reaching them",
    f"top within {abs(rq_samples.max()-lam[-1])/abs(lam[-1]):.2e}, bottom within "
    f"{abs(rq_samples.min()-lam[0])/abs(lam[0]):.2e}; a tighter gate here would "
    "be testing the random number generator, not the theorem",
)
validate.close(
    rq_at_eigs, lam,
    "while rho(q_i) = lambda_i exactly, for every i (Eq. 4)",
    rtol=0.0, atol=TOL,
)
validate.check(
    grads.max() < TOL and grad_random > 1.0,
    "every eigenvector is a stationary point of rho, interior ones included (Eq. 5)",
    f"largest ||grad rho|| over the five eigenvectors is {grads.max():.2e}, "
    f"against {grad_random:.4f} at a random unit vector; the three interior "
    "eigenvectors are saddle points, stationary but not extremal",
)
validate.close(
    np.array([rq_circle.max(), rq_circle.min()]), np.array([lam2[1], lam2[0]]),
    "and on the circle the quotient touches both eigenvalues of S_2",
    rtol=0.0, atol=1e-6,
)

## Exercise 4 — Courant–Fischer, checked from both sides

{eq}`eq-spec-minmax` characterises *every* eigenvalue, not just the extremes,
and it does so without naming an eigenvector. That is what makes it the engine
behind the two results that follow.

It is a min–max, so verifying it numerically means verifying two different
things. The **inequality** is that every $k$-dimensional subspace has
$\max_{\mathbf{x}\in V}\rho(\mathbf{x}) \ge \lambda_k$ — a statement about all
subspaces, which random sampling can only support, never establish, but which
random sampling can *refute* if it is false. The **attainment** is that the
specific subspace $\operatorname{span}\{\mathbf{q}_1,\dots,\mathbf{q}_k\}$
achieves $\lambda_k$ exactly, which is checkable to machine precision. The two
together are the theorem.

For a subspace with orthonormal basis in the columns of $B$ (shape `(5, k)`),
the maximum of $\rho$ over it is the largest eigenvalue of the $k\times k$
compression $B^{\top}\!SB$, which is why the whole thing is computable at all.

**Part a)** Write `max_rayleigh_on(B)` returning
`float(np.linalg.eigvalsh(B.T @ S @ B).max())` for an orthonormal `B`, and
confirm on `B = Q[:, :1]` that it returns $\lambda_1$ to $10^{-14}$.

**Part b)** Confirm **attainment**: for $k = 1, \dots, 5$, take
`B = Q[:, :k]`, which is orthonormal by Exercise 1, and confirm
`max_rayleigh_on(B)` equals `lam[k-1]` to $10^{-14}$. The eigenvector span is
the minimising subspace of {eq}`eq-spec-minmax`.

**Part c)** Confirm the **inequality** by sampling. For each $k$, draw 4000
random $5\times k$ Gaussian matrices, orthonormalise each with
`np.linalg.qr`, and confirm that *every* one gives
`max_rayleigh_on(B) >= lam[k-1] - 1e-12`. Not one of the 20,000 subspaces
tested may violate it; a single counterexample would disprove
{eq}`eq-spec-minmax`.

**Part d)** Report the minimum over the sampled subspaces for each $k$ beside
$\lambda_k$. The gaps are $0.012$, $0.115$, $0.033$, $0.000$, $0.000$ — random
subspaces get close for $k = 4, 5$ and less close in the middle, for the same
dimensional reason as Exercise 3. Confirm every sampled minimum is at least
$\lambda_k$ and within $0.5$ of it.

**Part e)** Confirm the *max–min* form is equivalent, by checking that
$\lambda_k$ is also $\max$ over $(n-k+1)$-dimensional subspaces of the
**minimum** of $\rho$, attained on
$\operatorname{span}\{\mathbf{q}_k,\dots,\mathbf{q}_n\}$. Take
`B = Q[:, k-1:]` and confirm `np.linalg.eigvalsh(B.T @ S @ B).min()` equals
`lam[k-1]` to $10^{-14}$ for every $k$.

In [ ]:
# (solution hidden on the public site)


### Validation 4

The two halves are checked by two different methods, and neither could replace
the other: attainment is exact and machine-checkable, the inequality is a
universal claim that sampling can only fail to refute. Counting *violations*
rather than measuring closeness is the right use of random subspaces here.

In [ ]:
validate.close(
    attain, lam,
    "span(q_1..q_k) attains lambda_k exactly: the minimising subspace (Eq. 6)",
    rtol=0.0, atol=TOL,
)
validate.close(
    attain_min, lam,
    "and the equivalent max-min form attains it on span(q_k..q_n) (Eq. 6)",
    rtol=0.0, atol=TOL,
)
validate.check(
    violations == 0,
    f"none of {5 * N_SUB} random subspaces violates the min-max inequality",
    "every sampled k-dimensional subspace has max Rayleigh quotient at least "
    "lambda_k; one counterexample would disprove Eq. 6, and there were none",
)
validate.check(
    bool(np.all(sampled_min >= lam - 1e-12))
    and bool(np.all(sampled_min - lam < 0.5)),
    "and the sampled minima approach lambda_k from above",
    f"gaps {[f'{g:.4f}' for g in (sampled_min - lam)]}: closer for k = 4, 5 than "
    "in the middle, for the same dimensional reason as Exercise 3",
)

## Exercise 5 — Cauchy interlacing

{eq}`eq-spec-interlacing` is one of those results that sounds like it needs a
hypothesis and does not. Delete the last row and column of *any* symmetric
matrix; the $n-1$ eigenvalues of what remains sit one in each gap of the
original $n$. No genericity assumption, no condition on the entries.

It follows from Courant–Fischer in a line: the $k$-dimensional subspaces of
$\mathbb{R}^{n-1}$ (viewed inside $\mathbb{R}^n$ as those with last coordinate
zero) are a *subset* of the $k$-dimensional subspaces of $\mathbb{R}^n$, so the
minimum in {eq}`eq-spec-minmax` over the smaller collection can only be larger:
$\lambda_k(A_{n-1}) \ge \lambda_k(A_n)$. Applying the same argument to $-A$
gives the other side.

**Part a)** Build the nested leading submatrices `S[:n, :n]` for
$n = 1, \dots, 5$ and compute `np.linalg.eigvalsh` of each. Print the five
spectra.

**Part b)** Check {eq}`eq-spec-interlacing` for every consecutive pair: with
`a = eigvalsh(S[:n, :n])` and `b = eigvalsh(S[:n-1, :n-1])`, confirm
`a[k] <= b[k] <= a[k+1]` for all $k = 0, \dots, n-2$, to $10^{-12}$. Report a
per-$n$ verdict and confirm all four hold.

**Part c)** Count the checks. Interlacing gives $n-1$ inequality pairs at each
level, so across $n = 2, 3, 4, 5$ there are $1 + 2 + 3 + 4 = 10$ pairs and 20
individual inequalities. Confirm all 20 hold, and report the tightest margin —
the smallest value of $\min(b_k - a_k,\, a_{k+1} - b_k)$ over all of them.

It is $2.4\times10^{-5}$, at $n = 5$, $k = 2$: the second eigenvalue of $S$ is
$3.3150379$ and the second eigenvalue of its $4\times4$ leading submatrix is
$3.3150616$. The inequality holds by a margin eleven orders above rounding and
five orders below the spectrum's own spread of $7.43$. This is a good place for
a theorem to be tested — a comfortable margin would prove less.

**Part d)** Confirm the immediate corollary: the extreme eigenvalues of a
leading submatrix are trapped by those of the whole, so
$\lambda_1(S) \le \lambda_1(S_{n-1})$ and $\lambda_{n-1}(S_{n-1}) \le
\lambda_n(S)$. Check both at $n = 5$ to $10^{-12}$.

**Part e)** Draw the ladder: plot each submatrix's spectrum as a row of
markers on a common horizontal axis, with $n$ increasing upward, so the
nesting is visible as each row's points falling into the gaps of the row above.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 5

Every one of the 20 inequalities is checked individually rather than in
aggregate, and the tightest margin is reported, because a theorem that holds
with a margin of $0.016$ is being tested meaningfully while one that holds with
a margin of $10^{-15}$ would not be.

In [ ]:
validate.check(
    all(interlace_ok),
    "Cauchy interlacing holds at every level of the nested submatrices (Eq. 7)",
    f"{n_pairs} interlacing pairs and {2*n_pairs} individual inequalities, all "
    f"satisfied; tightest margin {min(margins):.6f}",
)
validate.check(
    1e-6 < min(margins) < 1e-3,
    "and one pair is nearly tight, which is where a theorem is worth testing",
    f"the tightest of the {2*n_pairs} inequalities has margin {min(margins):.3e} "
    f"-- eleven orders above rounding, five below the spectrum's spread of "
    f"{spectra[4][-1]-spectra[4][0]:.3f}",
)
validate.check(
    low_ok and high_ok,
    "the corollary follows: a submatrix's extremes are trapped by the whole's",
    f"lambda_1(S) = {spectra[4][0]:.6f} <= lambda_1(S_4) = {spectra[3][0]:.6f}, "
    f"and lambda_4(S_4) = {spectra[3][-1]:.6f} <= lambda_5(S) = "
    f"{spectra[4][-1]:.6f}",
)

## Exercise 6 — Weyl, and a condition number equal to 1

{eq}`eq-spec-weyl` is the reason to trust `eigh`. Perturbing a symmetric matrix
by $E$ moves *no* eigenvalue further than $\|E\|_2$ — the eigenvalues are
1-Lipschitz functions of the matrix. In the language of
[§0.2](../00-machine/floating-point.ipynb) the symmetric eigenvalue problem has
condition number 1: a relative input error of $\varepsilon$ produces an
absolute output error of at most $\varepsilon\|A\|$, and no worse.

Set that against everything else in this course. Solving $A\mathbf{x} =
\mathbf{b}$ amplifies by $\kappa(A)$; forming the normal equations amplifies by
$\kappa(A)^2$; deconvolution amplified by $10^{16}$. Here the amplification
factor is 1, and it is 1 because of symmetry alone.

**Part a)** Draw 100 random symmetric perturbations as
`E = 0.3 * rng.standard_normal((5, 5))` symmetrised by `(E + E.T) / 2`. For
each, compute `np.linalg.eigvalsh(S + E)` and the ratio
$\max_k|\lambda_k(S+E) - \lambda_k(S)| \big/ \|E\|_2$.

**Part b)** Confirm {eq}`eq-spec-weyl`: every one of the 100 ratios is at most
1, to $10^{-12}$. Report the worst, which is $0.905$ — close enough to 1 that
the bound is being genuinely tested rather than trivially satisfied.

**Part c)** Confirm the bound is *sharp* by constructing a case that nearly
attains it: take `E = 0.3 * np.outer(Q[:, -1], Q[:, -1])`, a perturbation
aligned with the top eigenvector. Confirm $\|E\|_2 = 0.3$ and that
$\lambda_5(S+E) - \lambda_5(S)$ equals $0.3$ to $10^{-12}$: the bound is
attained exactly, so no constant smaller than 1 would do.

**Part d)** Contrast with the non-symmetric case. Take the defective
$J = \left[\begin{smallmatrix}1&1\\0&1\end{smallmatrix}\right]$ from
[§3.1](eigenvalues-diagonalization.ipynb) and perturb it by
$\delta = 10^{-8}$ in the lower-left corner. Report $\|E\|_2 = 10^{-8}$ and the
resulting eigenvalue movement, which is $10^{-4}$ — a factor of $10^{4}$
**larger** than the perturbation. Confirm the ratio exceeds $1000$. Weyl is a
statement about symmetric matrices and it fails spectacularly without that
hypothesis; the general bound involves $\operatorname{cond}(X)$, which is what
[§3.5](schur-jordan-nonnormality.ipynb) develops.

**Part e)** Confirm the scaling of that failure: for
$\delta = 10^{-4}, 10^{-6}, 10^{-8}, 10^{-10}$ the eigenvalue movement should
go like $\sqrt{\delta}$, since a $2\times2$ Jordan block splits its double
eigenvalue as $1 \pm \sqrt{\delta}$. Fit $\log(\text{movement})$ against
$\log\delta$ with `np.polyfit` and confirm the slope is $0.5$ to within 1%.

In [ ]:
# (solution hidden on the public site)


### Validation 6

The bound is checked three ways: that it holds on random perturbations, that it
is *attained* by an aligned one (so no smaller constant would do), and that it
fails without symmetry. The third is what makes the first two informative — a
bound that held for every matrix would say nothing about symmetry.

In [ ]:
validate.check(
    bool((ratios <= 1.0 + 1e-12).all()),
    f"Weyl holds for all {N_PERT} random symmetric perturbations (Eq. 8)",
    f"worst ratio |delta lambda| / ||E||_2 = {ratios.max():.6f}, median "
    f"{np.median(ratios):.6f}: the symmetric eigenvalue problem has condition "
    "number 1",
)
validate.check(
    ratios.max() > 0.5,
    "and the worst case comes close enough to 1 to be testing the bound",
    f"{ratios.max():.6f}; a worst ratio of 0.01 would mean the perturbations "
    "were too gentle to say anything",
)
validate.close(
    np.array([shift]), np.array([0.3]),
    "an aligned perturbation attains it exactly: the constant 1 is sharp (Eq. 8)",
    rtol=0.0, atol=1e-12,
)
validate.check(
    moves[2] / deltas[2] > 1000.0,
    "while without symmetry it fails: a 1e-8 perturbation moves lambda by 1e-4",
    f"the defective J moves its eigenvalue {moves[2]:.3e} under a perturbation "
    f"of {deltas[2]:.0e}, a ratio of {moves[2]/deltas[2]:.2e}. Weyl is a "
    "theorem about symmetric matrices, not about matrices",
)
validate.close(
    np.array([jordan_slope]), np.array([0.5]),
    "and it fails as sqrt(delta), the 2x2 Jordan block's splitting rate",
    rtol=1e-2, atol=0.0,
)

## Exercise 7 — Gershgorin: localising a spectrum without computing it

{eq}`eq-spec-gershgorin` gives a region containing every eigenvalue, from the
entries alone. There is no factorization, no iteration, and no arithmetic
beyond $n^2$ absolute values and additions. For a symmetric matrix the discs
are intervals on the real line, since the eigenvalues are real.

The theorem earns its keep when the discs are *disjoint*: a connected component
formed from $m$ discs contains exactly $m$ eigenvalues, so disjoint discs
localise each eigenvalue individually. That is why diagonally dominant matrices
are so much easier to work with than general ones, and it is the cheapest
available proof that a matrix is nonsingular — if no disc contains 0, then 0 is
not an eigenvalue.

**Part a)** For `S`, compute the centres `np.diag(S)` and the radii
`np.abs(S).sum(1) - np.abs(np.diag(S))`. Report both. The centres are
$4, 5, 6, 3, 7$ and the radii $4, 4, 3, 3, 4$.

**Part b)** Confirm {eq}`eq-spec-gershgorin`: every eigenvalue of `S` lies in
at least one disc, checked as $\min_i|\lambda - c_i| - R_i \le 10^{-12}$ for
each $\lambda$. Report which disc each eigenvalue falls in.

**Part c)** Note that the bound is loose here. The discs for `S` overlap into
a single interval $[0, 11]$, so the theorem localises the whole spectrum to
that interval and no eigenvalue individually. Report the union interval and
confirm the true spectrum $[1.742, 9.173]$ sits inside it.

**Part d)** Now the diagonally dominant `S_DOM`, whose diagonal is
$3, 10, 17, 24, 31$ with off-diagonal entries of magnitude $0.4$. Compute its
centres and radii, confirm the discs are pairwise disjoint by checking
$|c_i - c_j| > R_i + R_j$ for every $i \ne j$, and confirm each disc contains
**exactly one** eigenvalue. Report the largest $|\lambda_i - c_i|$, which is
$0.041$ against a largest radius of $1.6$.

**Part e)** Use it to prove nonsingularity without computing anything. Confirm
that no disc of `S_DOM` contains 0, hence 0 is not an eigenvalue, hence the
matrix is invertible — and check that against `np.linalg.det(S_DOM)`, which is
nonzero. Report the smallest distance from 0 to any disc, which is $1.8$: the
nearest disc is centred at 3 with radius $1.2$. Note how easily this fails —
lowering that first diagonal entry from 3 to 1 puts 0 inside the disc
$[-0.2, 2.2]$ and the argument evaporates, even though the matrix is still
perfectly invertible. Gershgorin proves nonsingularity when it can; it never
disproves it.

**Part f)** Draw both sets of discs on the complex plane with `la.gershgorin`,
with the true eigenvalues marked, so the overlapping and the disjoint cases can
be compared side by side.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 7

Containment is the theorem and is checked as such; the *usefulness* of the
bound is a separate matter, and reporting that the discs of `S` merge into one
interval is as much a part of the result as the containment itself. The
nonsingularity argument is included because it is the form in which Gershgorin
is most often actually used.

In [ ]:
validate.check(
    all(inside),
    "every eigenvalue of S lies in some Gershgorin disc (Eq. 9)",
    f"spectrum [{lam[0]:.4f}, {lam[-1]:.4f}] against discs centred "
    f"{centres.tolist()} with radii {radii.tolist()}, computed with no "
    "arithmetic beyond 25 absolute values",
)
validate.check(
    union_lo <= lam[0] and lam[-1] <= union_hi and (union_hi - union_lo) > 8.0,
    "though for S the bound is loose: the discs merge into one interval",
    f"[{union_lo:.1f}, {union_hi:.1f}] of width {union_hi-union_lo:.1f} against "
    f"a true spread of {lam[-1]-lam[0]:.4f}: overlapping discs localise the "
    "spectrum only as a whole",
)
validate.check(
    disjoint and counts == [1, 1, 1, 1, 1],
    "for S_DOM the discs are disjoint, so each holds exactly one eigenvalue",
    f"eigenvalues per disc {counts}; each lambda_i is within "
    f"{np.abs(np.sort(lam_dom) - np.sort(cd)).max():.4f} of its centre, against "
    f"a radius of {rd.max():.4f}",
)
validate.check(
    zero_dist > 0.0 and abs(np.linalg.det(S_DOM)) > 1e-6,
    "and since no disc reaches 0, S_DOM is invertible — proved by inspection",
    f"the nearest disc stops {zero_dist:.4f} short of the origin; "
    f"det(S_DOM) = {np.linalg.det(S_DOM):.4f} confirms it",
)

## Exercise 8 — The Rayleigh quotient converges quadratically, and the constant is the gap

[§3.1](eigenvalues-diagonalization.ipynb) ran power iteration on a
non-symmetric matrix and found the Rayleigh quotient converging at the *same*
linear rate as the vector, with a note that the famous quadratic rate needs
symmetry. This exercise settles that, and finds the constant exactly.

The argument is short enough to give here. Write the iterate as
$\mathbf{x} = \cos\theta\,\mathbf{q}_n + \sin\theta\,\mathbf{w}$ with
$\mathbf{w}$ a unit vector orthogonal to $\mathbf{q}_n$. Orthonormality of the
eigenvectors — available only under symmetry — makes the cross terms vanish, so

```{math}
:label: eq-spec-quadratic
\rho(\mathbf{x}) = \lambda_n\cos^2\theta + \rho(\mathbf{w})\sin^2\theta,
\qquad
\lambda_n - \rho(\mathbf{x}) = \bigl(\lambda_n - \rho(\mathbf{w})\bigr)\sin^2\theta .
```

The error in the eigenvalue is $O(\theta^2)$ where the error in the vector is
$O(\theta)$: that is the quadratic convergence. And since
$\|\mathbf{x} - \mathbf{q}_n\|^2 = 2(1 - \cos\theta) \approx \theta^2$ and
$\mathbf{w} \to \mathbf{q}_{n-1}$ as the iteration proceeds, the constant of
proportionality tends to $\lambda_n - \lambda_{n-1}$ — the **spectral gap**.

The matrix is $S_2 = \left[\begin{smallmatrix}6&1\\1&5\end{smallmatrix}\right]$
from Exercise 3, with eigenvalues $\lambda_1 = 4.381966$ and
$\lambda_2 = 6.618034$, so the gap is exactly $\sqrt5 = 2.2360680$.

**Part a)** Run power iteration {eq}`eq-eig-poweriter` on `S2` from
$\mathbf{x}_0 = (1, 0)$ for 25 steps, normalising with
`x / np.linalg.norm(x)` each time, recording the vector error
$\min(\|\mathbf{x}_k - \mathbf{q}_2\|, \|\mathbf{x}_k + \mathbf{q}_2\|)$ and
the Rayleigh error $|\rho(\mathbf{x}_k) - \lambda_2|$ at every step.

**Part b)** Report both errors at $k = 0, 5, 10, 15$. The vector error runs
$0.547, 7.9\times10^{-2}, 1.0\times10^{-2}, 1.3\times10^{-3}$ and the Rayleigh
error $0.618, 1.4\times10^{-2}, 2.2\times10^{-4}, 3.6\times10^{-6}$: the second
falls roughly as the square of the first.

**Part c)** Confirm the rates separately by fitting. Fit
$\log(\text{vector error})$ against $k$ over steps 5 to 20 and confirm the
slope is $\log|\lambda_1/\lambda_2| = \log 0.662125$ to within 2%. Then fit the
Rayleigh error the same way and confirm its slope is *twice* that, again to
within 2%.

**Part d)** Confirm the constant. Compute the ratio
$|\rho(\mathbf{x}_k) - \lambda_2| \big/ \|\mathbf{x}_k - \mathbf{q}_2\|^2$ for
$k = 3, \dots, 13$ and confirm it converges to $\lambda_2 - \lambda_1 = \sqrt5$
to $10^{-3}$. The quadratic convergence has an explicit constant and it is the
spectral gap — a wider gap gives faster convergence in both the vector and the
eigenvalue, which is the same fact twice.

**Part e)** Plot both error sequences on a log axis against $k$, with the
predicted slopes drawn as reference lines, so the factor-of-two in the
exponents is visible as a factor-of-two in the slopes.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 8

The two rates are fitted independently and their *ratio* is checked to be 2,
which is a stronger statement than either slope alone. The constant is then
checked against $\lambda_2 - \lambda_1$, an exact quantity, which turns "the
convergence is quadratic" into a prediction with a number in it.

In [ ]:
validate.close(
    np.array([slope_vec]), np.array([np.log(ratio2)]),
    "the eigenvector converges at |lambda_1/lambda_2|^k, as in 3.1 (Eq. 7 there)",
    rtol=2e-2, atol=0.0,
)
validate.close(
    np.array([slope_rq]), np.array([2.0 * np.log(ratio2)]),
    "but the Rayleigh quotient converges at TWICE that rate (Eq. 10)",
    rtol=2e-2, atol=0.0,
)
validate.close(
    np.array([slope_rq / slope_vec]), np.array([2.0]),
    "the ratio of the two fitted slopes is 2: quadratic convergence",
    rtol=2e-2, atol=0.0,
)
validate.close(
    np.array([const[-1]]), np.array([lam_gap]),
    "and the constant is the spectral gap lambda_2 - lambda_1 = sqrt(5) (Eq. 10)",
    rtol=0.0, atol=1e-3,
)
validate.check(
    rq_err[15] / vec_err[15] < 0.01,
    "so by step 15 the eigenvalue error is 350x smaller than the vector's",
    f"Rayleigh error {rq_err[15]:.3e} against vector error {vec_err[15]:.3e}, a "
    f"ratio of {rq_err[15]/vec_err[15]:.2e}, where at step 5 the ratio was "
    f"{rq_err[5]/vec_err[5]:.2e}. In 3.1, without symmetry, the two fell at the "
    "same rate and this ratio would have been flat",
)

---
## Notebook summary

**Symmetry removes every difficulty at once.** For the $5\times5$ integer
matrix $S$, `eigh` returned real eigenvalues in ascending order
($1.7420, 3.3150, 3.8300, 6.9401, 9.1730$) with $Q^{\top}Q = I$ to
$9\times10^{-16}$, $\operatorname{cond}(Q) = 1$ to $10^{-15}$, and
$Q\Lambda Q^{\top} = S$ to $8\times10^{-15}$ — with no inverse formed anywhere,
because $Q^{-1} = Q^{\top}$. Against
[§3.1](eigenvalues-diagonalization.ipynb)'s $\operatorname{cond}(X) = 2.51$ for
a non-symmetric matrix and $9\times10^{15}$ for a defective one, that 1 is the
whole theorem in a number.

**A symmetric matrix is a list of directions with numbers attached.** The five
rank-1 projectors were idempotent to $4\times10^{-16}$, mutually orthogonal to
$10^{-16}$, summed to $I$ to $7\times10^{-16}$, and reproduced $S$ to
$8\times10^{-15}$ when weighted by the eigenvalues. The top-$k$ partial sums
fell $5.215, 2.611, 2.377, 0.748, 0$, exact only at $k = n$.

**The Rayleigh quotient is an optimisation whose stationary points are the
eigenvectors.** All $10^{5}$ random unit vectors gave values inside
$[\lambda_1, \lambda_5]$ — a theorem, checked as one — approaching the top to
$1.3\times10^{-3}$ relative and the bottom to $1.4\times10^{-2}$, while
$\rho(\mathbf{q}_i) = \lambda_i$ *exactly* to $10^{-15}$. The gradient vanished
to $10^{-14}$ at all five eigenvectors, including the three interior saddle
points, against $6.2$ at a random direction.

**Courant–Fischer, checked from both sides.** The subspace
$\operatorname{span}\{\mathbf{q}_1,\dots,\mathbf{q}_k\}$ attained $\lambda_k$
to $10^{-15}$ for every $k$, and none of 20,000 random subspaces violated the
inequality — which is the only thing sampling can honestly establish about a
claim quantified over all subspaces.

**Interlacing and Weyl.** All 20 individual interlacing inequalities across the
nested submatrices held, with a tightest margin of $0.0163$. Weyl held for all
100 random symmetric perturbations with a worst ratio of $0.905$, and an
aligned perturbation $0.3\,\mathbf{q}_5\mathbf{q}_5^{\top}$ attained the bound
*exactly*, so the constant 1 is sharp. Without symmetry it fails outright: a
$10^{-8}$ perturbation of the defective $J$ moved its eigenvalue by $10^{-4}$,
a ratio of $10^{4}$, at the fitted rate $\delta^{0.5}$.

**Gershgorin localises a spectrum for free.** Every eigenvalue of $S$ lay in
some disc, though the discs merged into $[0, 11]$ against a true spread of
$7.43$. For the diagonally dominant $S_{\text{DOM}}$ the discs were pairwise
disjoint, each held exactly one eigenvalue within $0.041$ of its centre, and
since none reached 0 the matrix is invertible by inspection.

**And the loose end from [§3.1](eigenvalues-diagonalization.ipynb) is
settled.** On a symmetric matrix the Rayleigh quotient converges at
**twice** the exponential rate of the eigenvector — fitted slopes in ratio
$2.00$ — and the constant of proportionality converges to
$\lambda_2 - \lambda_1 = \sqrt5$, the spectral gap.

**Methods introduced.** `np.linalg.eigh` and `eigvalsh`, the spectral
projector sum, the Rayleigh quotient and its gradient, subspace compression
$B^{\top}\!AB$ as the computable form of a min–max, nested leading submatrices,
`ecp.linalg.gershgorin`, and the aligned perturbation as a way to test whether
a bound is sharp.

## Outlook

- **When all the eigenvalues are positive.** Symmetry gives real eigenvalues;
  requiring them positive gives *positive definiteness*, which turns
  $\mathbf{x}^{\top}\!A\mathbf{x}$ into a genuine notion of squared length,
  makes the quadratic form an ellipse rather than a saddle, and licenses the
  Cholesky factorization. [§3.3](positive-definite-cholesky.ipynb) develops it,
  and it underwrites every covariance matrix and every kernel in the volumes
  that follow.
- **The same theorem over $\mathbb{C}$.** Replacing $A = A^{\top}$ by
  $A = A^{*}$ gives Hermitian matrices, with the identical conclusion and a
  physical interpretation: observables in quantum mechanics are Hermitian
  *because* the spectral theorem guarantees real eigenvalues.
  [§3.4](hermitian-unitary-normal.ipynb) makes that trip.
- **The largest class for which any of this survives.** Symmetry is
  sufficient but not necessary. The exact condition for a unitary
  diagonalization is **normality**, $AA^{*} = A^{*}A$, which admits complex
  eigenvalues while keeping orthogonal eigenvectors — and everything outside
  that class is [§3.5](schur-jordan-nonnormality.ipynb)'s subject.
- **Interlacing as an algorithm.** {eq}`eq-spec-interlacing` is not only a
  theorem; running it in reverse on a tridiagonal matrix counts eigenvalues
  below a shift and gives the bisection method, which computes any single
  eigenvalue of a symmetric matrix to full accuracy without touching the
  others. [§5.2](../05-numerical/eigenvalue-algorithms.ipynb) builds it.

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()